In [2]:
import numpy as np
from tensorflow.keras.models import load_model
import tensorflow as tf

# === Simple eval for Model 0 (binary) using M1 eval style ===
# - Loads Router_test and RouterOCC_test, concatenates
# - Maps raw labels {0,1,2}->0 (Non-Error), {3}->1 (Error)
# - Evaluates model0_cnn on the combined test set

# --- paths ---
MODEL_PATH        = "../../../models/Model0/model0_cnn.keras"
X_TEST_PATH       = "../../../data/final test/Router_final test_X.npy"
Y_TEST_PATH       = "../../../data/final test/Router_final test_y.npy"
X_TEST_PATH_OCC   = "../../../data/final test/RouterOCC_final test_X.npy"
Y_TEST_PATH_OCC   = "../../../data/final test/RouterOCC_final test_y.npy"

# --- re-declare the lambda function used in training ---
def deltas_fn(t):
    d = t[:, 1:, :] - t[:, :-1, :]
    zero = tf.zeros_like(d[:, :1, :])
    return tf.concat([zero, d], axis=1)

# --- load data ---
X1 = np.load(X_TEST_PATH).astype(np.float32)        # (N1,4,4)
y1_raw = np.load(Y_TEST_PATH).astype(np.int64)      # (N1,)

X2 = np.load(X_TEST_PATH_OCC).astype(np.float32)    # (N2,4,4)
y2_raw = np.load(Y_TEST_PATH_OCC).astype(np.int64)  # (N2,)

# combine
X_test = np.concatenate([X1, X2], axis=0)
y_test_raw = np.concatenate([y1_raw, y2_raw], axis=0)

# map to binary: {0,1,2}->0, {3}->1
if not np.all(np.isin(y_test_raw, [0,1,2,3])):
    bad = np.unique(y_test_raw[~np.isin(y_test_raw, [0,1,2,3])])
    raise ValueError(f"Unexpected labels in test set: {bad}")

y_test = (y_test_raw == 3).astype(np.int64)

# --- load & eval ---
model = load_model(MODEL_PATH, custom_objects={"deltas_fn": deltas_fn})
loss, acc = model.evaluate(X_test, y_test, verbose=1)
print(f"✅ M0 Final Test accuracy (Router + RouterOCC, binary): {acc:.4f}")


1219/1219 ━━━━━━━━━━━━━━━━━━━━ 1s 810us/step - accuracy: 0.9997 - loss: 7.4737e-04
✅ M0 Final Test accuracy (Router + RouterOCC, binary): 0.9997
